## **PORTAFOLIO ACADÉMICO DE BI:**
# **PROYECTO INMOBILIARIO — CIUDAD DE LA COSTA**
# **Infraestructura de Historización — Preparando el seguimiento en el tiempo**
---
## Módulo 5:
* Introducción: De la vista viva al histórico
* Conexión y recuperación de Datos del Módulo 1
* Carga del snapshot histórico
* Vista de variación mensual

---
# **Introducción**
## **De la vista viva al histórico:**

Con los cuatro módulos académicos cerrados el proyecto entra en su etapa de infraestructura empresarial. Este módulo no responde una pregunta de negocio como los anteriores: prepara la base técnica que va a sostener el panel de control y el dashboard de seguimiento.

Hasta ahora trabajamos exclusivamente sobre `v_ciudad_de_la_costa`, una vista que siempre muestra el estado **actual** del mercado. Cada vez que se actualiza el Google Sheet fuente y se vuelve a correr el ETL del Módulo 1, los datos anteriores se pierden: la vista no guarda memoria de sí misma.

Esto es correcto para el análisis exploratorio, pero insuficiente para responder una pregunta que sí le va a importar a la dirección: **¿cómo cambió el mercado este mes respecto al anterior?**

En este módulo resolvemos esa limitación construyendo una **tabla histórica**: cada vez que se actualiza la fuente se guarda una fotografía (snapshot) con fecha, sin sobreescribir las fotografías anteriores. Sobre esa tabla histórica se construye después una vista de variación mensual, que va a ser la fuente de datos directa del panel de control.

**Los mismos límites metodológicos documentados desde el Módulo 1 siguen aplicando:** la exclusión de terrenos en el cálculo de precio de vivienda, el uso de mediana en vez de promedio, y las advertencias de completitud de datos por barrio.

---

### **Conexión y recuperación de Datos del Módulo 1:**

Para dar inicio a este módulo vinculamos el cuaderno de trabajo con la infraestructura de datos que ya dejamos consolidada, ejecutando las mismas dos celdas clave que venimos usando desde el Módulo 2 para reestablecer la conexión de forma segura:

1. **Autenticación de la Cuenta:** En la primera celda, validamos nuestras credenciales de Google para autorizar el acceso desde este nuevo entorno de Colab.
2. **Inicialización del Cliente y Carga de Datos:** En la segunda celda, nos conectamos formalmente a nuestro proyecto de BigQuery (`proyectosuy`) para llamar directamente a la Vista unificada que construimos y limpiamos previamente, asegurando que trabajaremos exactamente con el mismo set de datos validado.

In [ ]:
# Autenticamos la cuenta de Google en el nuevo cuaderno
from google.colab import auth
auth.authenticate_user()

print("Autenticación completada con éxito")

Autenticación completada con éxito


In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd

# 1. Nos autenticamos asegurando el proyecto destino
auth.authenticate_user(project_id="proyectosuy")

# 2. Inicializamos el cliente pasándole explícitamente el ID en el constructor
client = bigquery.Client(project="proyectosuy")

# 3. Definimos la consulta
query_toda_la_base = """
    SELECT *
    FROM `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
"""

# 4. Cargamos todo el Data Frame completo
# Pasamos el project_id también en el método de ejecución por seguridad
df_completo = client.query(query_toda_la_base, project="proyectosuy").to_dataframe()

# Verificamos que cargaron todas las columnas y filas
print(f"Base de datos cargada. Total de propiedades: {len(df_completo)}")
df_completo.head()

Base de datos cargada. Total de propiedades: 555


,id,publicacion,finalizacion,operacion,tipo_inmueble,moneda,precio,ubicacion,zona,pisos,...,banos,cochera,parrillero,jardin,piscina,mts2_terreno,mts2_edificado,antiguedad,gastos_comunes_UYU,detalles
0,1,2025-06-01,NaT,Venta,Casa,U$S,289000.0,Solymar,Sur,2,...,4,1,True,True,False,364,139.0,20,NaN,Construcción sólida
1,2,2026-05-07,NaT,Venta,Casa,U$S,250000.0,Solymar,Sur,1,...,2,1,True,True,False,310,95.0,1,NaN,Construcción sólida
2,3,2026-05-05,NaT,Venta,Complejo,U$S,187000.0,Solymar,Norte,2,...,2,2,True,False,False,148,78.2,0,NaN,Próximo a Car One
3,4,2026-01-12,NaT,Venta,Casa,U$S,265000.0,Solymar,Sur,1,...,2,1,True,True,False,527,210.0,30,NaN,Casa independiente
4,5,2026-04-02,NaT,Venta,Casa,U$S,185000.0,Lagomar,Norte,1,...,1,1,True,True,False,200,85.0,1,NaN,Próximo a Almenara Mall


## **Carga del snapshot histórico**

Con `df_completo` ya cargado, el siguiente paso es guardar una fotografía con fecha de este estado del mercado en una tabla histórica (`historico_ciudad_de_la_costa`), que va a ir acumulando una fila por snapshot cada vez que se actualice la fuente.

**Nota de infraestructura:** el proyecto de BigQuery de este trabajo corre en el nivel gratuito (sandbox), que no tiene una cuenta de facturación asociada. En ese modo, las consultas de lectura (`SELECT`) y de definición de esquema están permitidas, pero las consultas de escritura vía SQL (`INSERT`, `UPDATE`) no lo están. Por eso la carga se hace con un **load job**: el mecanismo nativo del cliente de Python para subir un DataFrame de pandas directo a una tabla, que sí está disponible en el nivel gratuito. Con `autodetect=True`, BigQuery además infiere el tipo de cada columna directamente desde los datos reales, en vez de que lo declaremos a mano.

La celda de código de abajo cumple dos roles a la vez, sin necesidad de pasos separados:
* **La primera vez que se corre:** como la tabla todavía no existe, el load job la crea automáticamente con el esquema inferido y carga el primer snapshot.
* **Cada vez que se vuelve a correr (con datos actualizados):** chequea si ya existe un snapshot con la fecha de hoy; si no existe, agrega uno nuevo sin tocar los anteriores, y si ya existe, no hace nada, para evitar duplicados.

In [ ]:
# =============================================================================
# CARGA DEL SNAPSHOT HISTÓRICO (crea la tabla la primera vez, agrega la
# fotografía del día en cada corrida posterior)
# =============================================================================

import datetime
from google.api_core.exceptions import NotFound

tabla_historica = "proyectosuy.mercado_inmobiliario.historico_ciudad_de_la_costa"

# 1. Chequeamos si la tabla ya existe
try:
    client.get_table(tabla_historica)
    tabla_existe = True
except NotFound:
    tabla_existe = False

# 2. Si ya existe, chequeamos si ya se cargó un snapshot hoy (evita duplicados)
cargar_snapshot = True
if tabla_existe:
    query_check = f"""
    SELECT COUNT(*) AS total
    FROM `{tabla_historica}`
    WHERE fecha_snapshot = CURRENT_DATE()
    """
    ya_existe_hoy = client.query(query_check).to_dataframe()["total"][0]
    cargar_snapshot = ya_existe_hoy == 0

# 3. Cargamos el snapshot solo si corresponde
if cargar_snapshot:
    df_snapshot = df_completo.copy()
    df_snapshot.insert(0, "fecha_snapshot", datetime.date.today())

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND",
        autodetect=True,
    )

    load_job = client.load_table_from_dataframe(df_snapshot, tabla_historica, job_config=job_config)
    load_job.result()

    print(f"Snapshot cargado con éxito: {len(df_snapshot)} filas.")
else:
    print("Ya existe un snapshot de hoy. No se vuelve a cargar para evitar duplicados.")

Snapshot cargado con éxito: 555 filas.


## **Vista de variación mensual**

Con al menos un snapshot ya guardado en `historico_ciudad_de_la_costa`, construimos la vista `v_variacion_mensual`: agrupa los snapshots por barrio y por mes, calcula el stock de viviendas y el precio mediana de cada combinación, y compara cada mes contra el anterior usando la función de ventana `LAG`.

Esta vista no se vuelve a tocar en las próximas cargas: como es una `VIEW` y no una tabla, se recalcula sola cada vez que se consulta, incorporando automáticamente cualquier snapshot nuevo que se agregue más adelante.

**Nota de lectura para esta primera corrida:** como el proyecto recién tiene un snapshot cargado, la columna `variacion_pct` va a aparecer vacía (`NULL`) para todos los barrios, no hay un mes anterior todavía con el cual comparar. La vista va a empezar a mostrar variación real recién a partir del segundo snapshot mensual que se cargue.

Se mantienen los mismos criterios metodológicos del resto del proyecto: se excluye `terreno` del cálculo de precio de vivienda, y se usa mediana (`APPROX_QUANTILES`) en vez de promedio, para no distorsionar la lectura con *outliers*.

In [ ]:
# =============================================================================
# VISTA DE VARIACIÓN MENSUAL:
# =============================================================================

query_vista_variacion = """
CREATE OR REPLACE VIEW `proyectosuy.mercado_inmobiliario.v_variacion_mensual` AS
WITH mensual AS (
  SELECT
    DATE_TRUNC(fecha_snapshot, MONTH) AS mes,
    ubicacion AS Barrio,
    COUNTIF(LOWER(operacion) = 'venta' AND LOWER(tipo_inmueble) != 'terreno') AS stock_vivienda,
    APPROX_QUANTILES(
      IF(LOWER(operacion) = 'venta' AND LOWER(tipo_inmueble) != 'terreno', precio, NULL), 2
    )[OFFSET(1)] AS precio_mediana
  FROM `proyectosuy.mercado_inmobiliario.historico_ciudad_de_la_costa`
  GROUP BY mes, Barrio
)
SELECT
  Barrio,
  mes,
  stock_vivienda,
  precio_mediana,
  LAG(precio_mediana) OVER (PARTITION BY Barrio ORDER BY mes) AS precio_mediana_mes_anterior,
  SAFE_DIVIDE(
    precio_mediana - LAG(precio_mediana) OVER (PARTITION BY Barrio ORDER BY mes),
    LAG(precio_mediana) OVER (PARTITION BY Barrio ORDER BY mes)
  ) AS variacion_pct
FROM mensual
ORDER BY Barrio, mes;
"""

client.query(query_vista_variacion).result()
print("Vista v_variacion_mensual creada con éxito.")

query_verificacion = "SELECT * FROM `proyectosuy.mercado_inmobiliario.v_variacion_mensual` ORDER BY Barrio"
client.query(query_verificacion).to_dataframe()

Vista v_variacion_mensual creada con éxito.


,Barrio,mes,stock_vivienda,precio_mediana,precio_mediana_mes_anterior,variacion_pct
0,Carrasco,2026-07-01,21,295000.0,NaN,NaN
1,Lagomar,2026-07-01,25,270000.0,NaN,NaN
2,Pinar,2026-07-01,18,250000.0,NaN,NaN
3,Shangrila,2026-07-01,32,300000.0,NaN,NaN
4,Solymar,2026-07-01,183,258000.0,NaN,NaN
5,Tahona,2026-07-01,30,590000.0,NaN,NaN


---
## **Cierre del Módulo 5**

Con este módulo, el proyecto incorpora su primera pieza de infraestructura empresarial: una tabla histórica (`historico_ciudad_de_la_costa`) que acumula una fotografía fechada del mercado en cada actualización, y una vista de variación mensual (`v_variacion_mensual`) que calcula automáticamente el cambio de stock y precio mediana por barrio mes a mes, sin necesidad de recalcularse manualmente.

Con un único snapshot cargado, la columna de variación porcentual todavía no tiene valores que mostrar — eso va a empezar a poblarse a partir del segundo snapshot mensual, cuando se vuelva a correr la celda de carga con datos actualizados.

Esta vista queda lista para conectarse directamente como fuente de datos en Looker Studio, tanto para el dashboard interactivo como para el panel de control de seguimiento.